In [ ]:
import numpy as np

# Ear Shape (pointy=1), Face Shape (round=1), Whiskers (present=1)
X_train = np.array([[1, 1, 1],
                    [0, 0, 1],
                    [0, 1, 0],
                    [1, 0, 1],
                    [1, 1, 1],
                    [1, 1, 0],
                    [0, 0, 0],
                    [1, 1, 0],
                    [0, 1, 0],
                    [0, 1, 0]])
y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])
feature_names = ['Ear Shape', 'Face Shape', 'Whiskers']


def entropy(p):
    if p == 0 or p == 1:
        return 0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


def split_indices(X, node_indices, feature):
    left = [i for i in node_indices if X[i][feature] == 1]
    right = [i for i in node_indices if X[i][feature] == 0]
    return left, right


def information_gain(y, node_indices, left, right):
    w_left = len(left) / len(node_indices)
    w_right = len(right) / len(node_indices)
    h_node = entropy(y[node_indices].mean())
    h_split = w_left * entropy(y[left].mean()) + w_right * entropy(y[right].mean())
    return h_node - h_split


def gains_for_node(X, y, node_indices):
    gains = {}
    for f in range(X.shape[1]):
        left, right = split_indices(X, node_indices, f)
        if not left or not right:
            gains[f] = 0.0
        else:
            gains[f] = information_gain(y, node_indices, left, right)
    return gains


def build_tree(X, y, node_indices, depth=0, max_depth=2, min_gain=1e-9):
    p = y[node_indices].mean()
    prediction = int(p >= 0.5)
    if depth == max_depth or p in (0, 1):
        return {"leaf": True, "indices": node_indices, "prediction": prediction}
    gains = gains_for_node(X, y, node_indices)
    best_f = max(gains, key=gains.get)
    if gains[best_f] < min_gain:
        return {"leaf": True, "indices": node_indices, "prediction": prediction}
    left, right = split_indices(X, node_indices, best_f)
    return {
        "leaf": False,
        "feature": best_f,
        "gain": gains[best_f],
        "indices": node_indices,
        "left": build_tree(X, y, left, depth + 1, max_depth, min_gain),
        "right": build_tree(X, y, right, depth + 1, max_depth, min_gain),
    }


def print_tree(node, indent="", branch=""):
    label = "Cat" if node.get("prediction") == 1 else "Not cat"
    if node["leaf"]:
        print(f"{indent}{branch}-> {label}  (examples {node['indices']})")
    else:
        print(f"{indent}{branch}Split on {feature_names[node['feature']]} "
              f"(gain {node['gain']:.2f}, examples {node['indices']})")
        print_tree(node["left"], indent + "    ", "[=1] ")
        print_tree(node["right"], indent + "    ", "[=0] ")


def predict(node, x):
    while not node["leaf"]:
        node = node["left"] if x[node["feature"]] == 1 else node["right"]
    return node["prediction"]


if __name__ == "__main__":
    root = list(range(len(X_train)))

    print("Information gain at the root:")
    for f, g in gains_for_node(X_train, y_train, root).items():
        print(f"  {feature_names[f]}: {g:.2f}")

    print("\nFinal tree (max_depth=2):")
    tree = build_tree(X_train, y_train, root, max_depth=2)
    print_tree(tree)

    preds = [predict(tree, x) for x in X_train]
    print("\nPredictions:", preds)
    print("Actual:     ", list(y_train))